# Comprehensive Telecom SaaS Audit: Revenue, Retention, and Reliability
**Role:** Data Analytics Engineer  
**Objective:** To demonstrate full-stack data analytics capabilities by auditing a mock Hybrid CCaaS (Contact Center as a Service) dataset. 

### Project Overview
This notebook executes a 360-degree business audit using **DuckDB** for high-performance SQL execution over **Parquet** files, combined with **Pandas** for executive-ready data visualization. 

The analysis is broken down into four strategic pillars, answering critical business questions for different departments:
1. **Finance:** Are we losing money on specific international telecom routes? *(Negative Margin Detection)*
2. **Customer Success:** Which customers are paying for empty seats and are at high risk of churning? *(Seat Utilization & Ghost Accounts)*
3. **Sales:** Which accounts have high international usage and are ripe for a local DID upsell? *(Commercial Expansion)*
4. **Engineering:** Are our carrier networks stable, and are SIP failure rates within acceptable SLAs? *(Platform Health)*

In [ ]:
# install duckdb if you didn't do it before 

# !pip install duckdb

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt 

con =  duckdb.connect(database=':memory:')

In [2]:
con.execute ( """
    create table dim_customers as 
     select * from read_parquet('dim_customers.parquet') ; 

    create table fact_calls as 
    select * from read_parquet('fact_calls.parquet') ; 

"""
            )

# first analysis: The Unit Economics Audit
#### The Business Context:
In a Hybrid CCaaS model, looking only at Monthly Recurring Revenue (MRR) can be **deceptive**.

A customer paying for multiple seats might seem highly profitable, but if their outbound telecom usage heavily targets high-cost destinations (where Carrier COGS exceed our Customer Rate Card), they become a financial liability.

### The Objective:
To calculate the True Gross Profit of each account. This requires merging fixed SaaS subscription revenue with variable telecom usage, strictly accounting for the 2,000 free inbound minutes allowance.

In [69]:
revneus_analysis = con.execute(
    """
    WITH monthly_usage AS ( 
    SELECT 
        call_id , 
        customer_id , 
        timestamp , 
        date_trunc('month' , timestamp) as billing_month , 
        direction , 
        caller_id_country , 
        destination_country , 
        duration_minutes , 
        rate_price , 
        rate_cost , 

        -- we need to make sure that we handle the 2000 free inbound minutes 
        sum(case when direction = 'inbound' then duration_minutes else 0 end)
            over (partition by customer_id , billing_month order by timestamp 
            rows between unbounded PRECEDING and current row  ) as running_inbound_total 
            
    FROM fact_calls 
), 

revenue_calculation AS ( 
    SELECT * , 
        (duration_minutes * rate_cost) as call_cost , 
        CASE 
            WHEN direction = 'outbound' THEN (duration_minutes * rate_price)
            WHEN direction = 'inbound' THEN 
                CASE 
                    WHEN running_inbound_total <= 2000 THEN 0.0 
                    WHEN (running_inbound_total - duration_minutes) < 2000 AND running_inbound_total > 2000
                        THEN (running_inbound_total - 2000) * 0.02
                    ELSE duration_minutes * 0.02 
                END 
        END AS call_revenue
    FROM monthly_usage
),

monthly_aggregated_usage AS (
    select 
        customer_id,
        billing_month,
        sum(call_revenue) AS total_usage_revenue,
        sum(call_cost) AS total_carrier_cost
    from revenue_calculation
    group by customer_id, billing_month
),

-- Calculate the fixed SaaS monthy reccuring revenues from the Dimension table
customer_mrr AS (
    SELECT 
        customer_id,
        company_name,
        (paid_seats * seat_price) + 
        ((length(owned_dids) - length(replace(owned_dids, '|', '')) + 1) * 30) AS fixed_mrr
    FROM dim_customers
)


SELECT 
    c.company_name,
    CAST(u.billing_month AS DATE) AS billing_month,
    c.fixed_mrr AS subscription_revenue,
    round(u.total_usage_revenue, 2) AS usage_revenue,
    round(u.total_carrier_cost, 2) AS carrier_cost,
    
    round((c.fixed_mrr + u.total_usage_revenue - u.total_carrier_cost), 2) AS true_gross_profit
    
FROM monthly_aggregated_usage u
join customer_mrr c ON u.customer_id = c.customer_id
ORDER BY true_gross_profit ASC -- Sort worst to best to find the "Tunisia Trap"
LIMIT 10;
    
    """
).df()

In [71]:
revneus_analysis.head(10)

,company_name,billing_month,subscription_revenue,usage_revenue,carrier_cost,true_gross_profit
0,Tunisia Trders,2025-01-01,1110,7443.04,10420.26,-1867.22
1,Tunisia Trders,2025-03-01,1110,7232.88,10126.04,-1783.15
2,Tunisia Trders,2025-02-01,1110,6783.56,9496.98,-1603.42
3,Support Desk Co,2025-02-01,255,750.95,470.25,535.70
4,Support Desk Co,2025-01-01,255,774.15,481.17,547.98
5,Support Desk Co,2025-03-01,255,810.07,507.13,557.94
6,"Guzman, Hoffman and Baldwin",2025-02-01,570,645.61,513.97,701.64
7,"Guzman, Hoffman and Baldwin",2025-01-01,570,666.86,530.59,706.27
8,"Guzman, Hoffman and Baldwin",2025-03-01,570,737.61,576.59,731.01
9,"Hensley, Powell and David",2025-02-01,615,626.32,496.23,745.09


## Executive Insights: The Unit Economics Audit

### The Observation:
The data immediately surfaces "Tunisia Traders" as a critical outlier. While they contribute fixed monthly SaaS MRR, their specific routing behavior generates massive carrier costs per month, resulting in a severe net loss every billing cycle.
         
### The Business Impact: 
This is a classic "Negative Margin Trap." The telecom Cost of Goods Sold (COGS) for this specific route is acting as a silent revenue leak, effectively wiping out the pure SaaS profit generated by several healthy accounts.

### Strategic Recommendation: 
1. Immediate Action: Auto-trigger a rate-card renegotiation workflow for Tunisia Traders, or route their traffic through a cheaper wholesale carrier via Least Cost Routing (LCR).
2. Systemic Fix: Implement an automated "Margin Drop Alert" to notify the Account Management team whenever a customer's usage margin on any specific country route drops below 15%.

# Second analysis: Product Audit - Seat Utilization & Churn Risk
#### The Business Context:
In SaaS, Monthly Recurring Revenue (MRR) is only valuable if it is retained.

A "sold" seat does not equal an "active" seat.

When a customer pays for significantly more licenses than they use, they are at a high risk of churning once their finance team audits their software spend.

### The Objective:
To identify "Zombie Accounts" 

We will calculate the Seat Utilization Rate by comparing the legally contracted paid_seats from the billing system against the actual count of agentst that generating call records in the platform over a 30-day window.

In [11]:
# code cell 
zombies_df = con.execute("""
with monthly_active_aggents as (
select
customer_id ,
date_trunc('month', timestamp) as billing_month  , 
count(distinct agent_id) as active_agent_count , 
from fact_calls 
group by customer_id , billing_month
)
select 
c.company_name , 
cast(m.billing_month as date ) as billing_month  , 
c.paid_seats , 
m.active_agent_count , 
(c.paid_seats - m.active_agent_count) as ghost_seats , 
round((m.active_agent_count * 100.0 / c.paid_seats) , 2) as utilization_percentage , 
((c.paid_seats - m.active_agent_count)* c.seat_price) as wasted_customer_budget 

from dim_customers c 
join monthly_active_aggents m on c.customer_id = m.customer_id 
order by utilization_percentage asc 
""").df()


In [28]:
# Format the dictionary to handle percentages and currency , this cell using gemmini ^ _ ^
format_dict_churn = {
    'utilization_percentage': '{:.2f}%',
    'wasted_customer_budget': '${0:,.2f}'
}

# The perfect combination for your specific Pandas version
styled_df = (zombies_df.head(10).style
    .format(format_dict_churn)
    .map(lambda x: 'color: red; font-weight: bold;' if x < 50 else 'color: black;', subset=['utilization_percentage'])
    .set_caption("High Churn Risk: Accounts by Seat Utilization")
    .hide(axis="index"))

# Display the styled table
display(styled_df)

company_name,billing_month,paid_seats,active_agent_count,ghost_seats,utilization_percentage,wasted_customer_budget
Ghost Corp,2025-01-01 00:00:00,100,12,88,12.00%,"$3,960.00"
Ghost Corp,2025-02-01 00:00:00,100,12,88,12.00%,"$3,960.00"
Ghost Corp,2025-03-01 00:00:00,100,12,88,12.00%,"$3,960.00"
House-Glover,2025-01-01 00:00:00,34,27,7,79.41%,$350.00
House-Glover,2025-02-01 00:00:00,34,27,7,79.41%,$350.00
House-Glover,2025-03-01 00:00:00,34,27,7,79.41%,$350.00
Allen-Allen,2025-01-01 00:00:00,49,39,10,79.59%,$500.00
Allen-Allen,2025-03-01 00:00:00,49,39,10,79.59%,$500.00
Allen-Allen,2025-02-01 00:00:00,49,39,10,79.59%,$500.00
Harris-Walters,2025-01-01 00:00:00,10,8,2,80.00%,$90.00


## Executive Insights: The Churn Risk 

### The Observation:
The data immediately isolates "Ghost Corp" as a severe churn risk. They are contractually paying for 100 seats, but only 12 distinct agent IDs are actively generating traffic on the platform. This results in an abysmal 12% seat utilization rate.

### The Business Impact:
Ghost Corp is wasting $3,960 every single month on "Ghost Seats." While this artificially inflates Maqsam's short-term MRR, it guarantees a catastrophic churn event. As soon as Ghost Corp's finance team audits their SaaS spending, they will not just cancel the 88 unused seats (they will likely rip out the entire Maqsam contract out of frustration).

### Strategic Recommendation: 
The Customer Success team must **intervene immediately**. We should proactively contact Ghost Corp and offer a contract "right-sizing" (e.g., downgrading them to 15-20 seats). Sacrificing the unused seat revenue today builds massive customer trust, saves the core account, and protects the long-term Lifetime Value (LTV) of this client.

# Third Analysis: Missed Upsell Opportunities (DIDs)

**The Business Context:**
In international telecom, outbound calls are billed at a much higher rate when the caller does not own a local phone number (DID - Direct Inward Dialing) in the destination country. For example, a company in Jordan calling Saudi Arabia using a Jordanian caller ID pays peak international rates and suffers lower call pickup rates. 

**The Objective:**
To identify accounts with high-volume international outbound traffic to countries where they currently do not rent a local Maqsam phone number. This acts as a highly targeted lead list for the Sales team to pitch $30/month international DIDs, saving the customer money while increasing Maqsam's sticky Monthly Recurring Revenue (MRR).

In [15]:
# query 
upsell_opp_df = con.execute("""

with outbound_traffic as (
select 
customer_id , 
date_trunc('month',timestamp) as billing_month , 
caller_id_country , 
destination_country , 
count(call_id) as total_calls , 
sum(duration_minutes) as total_minutes , 
sum(duration_minutes * rate_price) as total_usage_spend
from fact_calls 
where direction = 'outbound' and caller_id_country != destination_country
group by 1 , 2 , 3, 4 )

select 
c.company_name , 
cast(o.billing_month as date) as billing_month , 
o.caller_id_country as calling_from , 
o.destination_country as calling_to , 
c.owned_dids as currently_owned_numbers , 
o.total_calls , 
round(o.total_minutes , 2) as total_minutes , 
round(o.total_usage_spend , 2) as customer_route_spend

from outbound_traffic o 
join dim_customers c on c.customer_id = o.customer_id 
where c.owned_dids not like '%' || o.destination_country || '%'
order by o.total_usage_spend desc 

""").df()


In [30]:
# Format the dataframe for executive presentation
format_dict_upsell = {
    'total_calls': '{:,.0f}',
    'total_minutes': '{:,.2f}',
    'customer_route_spend': '${0:,.2f}'
}

# Apply formatting and display the top 10 rows
display(upsell_opp_df.head(10).style
    .format(format_dict_upsell)
    .map(lambda x: 'background-color: #d4edda;' if x == 'Saudi Arabia' else '', subset=['calling_to'])
    .set_caption("Commercial Audit: High-Value International Routes without Local DIDs")
    .hide(axis="index"))

company_name,billing_month,calling_from,calling_to,currently_owned_numbers,total_calls,total_minutes,customer_route_spend
Tunisia Trders,2025-01-01 00:00:00,Jordan,Tunisia,jordan,"7,427","29,772.17","$7,443.04"
Tunisia Trders,2025-03-01 00:00:00,Jordan,Tunisia,jordan,"7,192","28,931.53","$7,232.88"
Tunisia Trders,2025-02-01 00:00:00,Jordan,Tunisia,jordan,"6,820","27,134.23","$6,783.56"
Gulf Exporters,2025-01-01 00:00:00,Jordan,Saudi Arabia,Jordan,"7,523","30,032.40","$5,405.83"
Gulf Exporters,2025-03-01 00:00:00,Jordan,Saudi Arabia,Jordan,"7,212","29,571.62","$5,322.89"
Gulf Exporters,2025-02-01 00:00:00,Jordan,Saudi Arabia,Jordan,"6,799","26,871.78","$4,836.92"
Ghost Corp,2025-01-01 00:00:00,UAE,UK,UAE,"1,079","4,634.73","$1,390.42"
Ghost Corp,2025-01-01 00:00:00,UAE,Saudi Arabia,UAE,"1,084","4,315.06","$1,294.52"
Ghost Corp,2025-03-01 00:00:00,UAE,Egypt,UAE,"1,029","4,232.28","$1,269.68"
Ghost Corp,2025-02-01 00:00:00,UAE,UK,UAE,982,"4,131.75","$1,239.53"


## Executive Insights: The Commercial Upsell Target

### The Observation:
The outbound traffic data surfaces Gulf Exporters as a massive commercial opportunity.

#### Current State: 
They are based in Jordan and currently only lease a Jordanian phone number (DID).

#### Routing Behavior: 
Despite this, they consistently generate over 7,000 outbound calls and $4,800 - $5,400 in usage spend specifically targeting Saudi Arabia every single month.

### The Business Impact:
Because Gulf Exporters does not own a local Saudi number, every call they make to Saudi Arabia is flagged as an international call.

### Financial Drain: 
This results in peak international billing rates for the customer.

### Performance Drop:
It drastically lowers their call connection rates, as end-users are significantly less likely to answer a foreign caller ID.

### Strategic Recommendation:
The Sales and Account Management team should immediately pitch Gulf Exporters a dedicated Saudi Arabian DID ($30/month).

#### Customer Value: 
Acquiring a local Saudi number will immediately increase their outbound call pickup rates and potentially qualify them for cheaper local-to-local routing tariffs.

#### Maqsam Value: 
This strategic upsell adds high-margin, sticky Monthly Recurring Revenue (MRR) to the account while locking the customer deeper into the Maqsam ecosystem.

# Analysis 4:Platform Health & SIP Errors

**The Business Context:**
In cloud communications, call quality and reliability are just as important as pricing. When a call fails, the telecom network returns a SIP (Session Initiation Protocol) response code. 
* **4xx Codes** (like 404 Not Found or 486 Busy) are usually user errors or normal behavior.
* **5xx Codes** (like 503 Service Unavailable or 500 Server Error) indicate a critical failure on our end or our carrier's end. 

**The Objective:**
To proactively identify any specific destination countries or routes that are experiencing an abnormally high rate of **5xx carrier failures**. If we find a failing route, our Voice Engineering team needs to immediately change the backend routing to a backup carrier before customers complain.

In [52]:
#Query 

platfor_health_df = con.execute("""
WITH total_calls_per_route AS (
    select 
        destination_country, 
        count(distinct call_id) AS total_calls_attempt,  
        count(distinct IF(sip_code >= 500, call_id, null )) AS critical_carrier_failures, 
        round((COALESCE(critical_carrier_failures, 0) * 100.0 / total_calls_attempt), 2) AS failure_rate_percentage
        
    from fact_calls 
   group by  1 
    
    having count (distinct call_id) > 1000
)

select * from total_calls_per_route 
order by failure_rate_percentage desc
limit 10 ; 
""").df()


In [54]:
# Format the dataframe for engineering presentation
format_dict_engineering = {
    'total_calls_attempt': '{:,.0f}',
    'critical_carrier_failures': '{:,.0f}',
    'failure_rate_percentage': '{:.2f}%'
}

# Apply formatting and highlight anything over 5% as a warning (even though none exist here!)
display(platfor_health_df.style
    .format(format_dict_engineering)
    .map(lambda x: 'background-color: #f8d7da; color: darkred;' if x > 5.0 else 'background-color: #d4edda; color: darkgreen;', subset=['failure_rate_percentage'])
    .set_caption("Engineering Audit: SIP 5xx Failure Rates by Route (Stable)")
    .hide(axis="index"))

destination_country,total_calls_attempt,critical_carrier_failures,failure_rate_percentage
Egypt,"87,377","1,913",2.19%
Jordan,"51,300","1,039",2.03%
Tunisia,"21,439",433,2.02%
Saudi Arabia,"136,213","2,710",1.99%
UAE,"116,712","2,297",1.97%
UK,"86,959","1,670",1.92%


## Executive Insights: Platform Stability & SIP Error Monitoring

### The Observation:
The engineering audit reveals that the platform's carrier connections are highly stable. Across all major routes, the SIP 5xx (Critical Carrier Failure) rate is hovering at a healthy ~2%, peaking at just 2.19% for traffic terminating in Egypt. There are no statistical anomalies indicating a localized carrier outage.


### The Business Impact:
Maqsam's core routing infrastructure is currently performing optimally. Customers are not experiencing "Service Unavailable" or "Server Error" network drops. Maintaining this sub-3% failure rate protects the brand's reputation for reliability and ensures we are safely complying with strict Enterprise Service Level Agreements (SLAs).



### Strategic Recommendation:
While this static monthly snapshot is healthy, telecom outages happen in minutes, not months.

The Action: The Data & Voice Engineering teams should collaborate to build a real-time streaming alert .

The Trigger: If the 5xx failure rate for any specific destination country spikes above 5% within a rolling 5-minute window, it should instantly trigger a PagerDuty alert.